# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
from dotenv import load_dotenv, find_dotenv

from lib.game_store import GameVectorStore

In [3]:
# The .env file lives in this folder (project/starter) and must contain:
# OPENAI_API_KEY="voc-..."
# OPENAI_BASE_URL="https://openai.vocareum.com/v1"
# TAVILY_API_KEY="tvly-..."
# See .env.example for the variable names.

In [4]:
# find_dotenv() finds the .env file in this folder (or a parent folder);
# override=True makes the values in .env win over anything already set.
load_dotenv(find_dotenv(), override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"
print("Environment variables loaded")

Environment variables loaded


### VectorDB Instance

The vector store is handled by a reusable manager, `GameVectorStore` in `lib/game_store.py`. It wraps:
- the persistent client: `chromadb.PersistentClient(path="chromadb")`
- the embedding function: `OpenAIEmbeddingFunction(api_key_env_var="OPENAI_API_KEY", api_base=os.getenv("OPENAI_BASE_URL"))`
- the `udaplay` collection, plus `load_games()` to index the JSON files and `search()` for semantic search

Part 2 opens the database through the same class, so path, collection name and embeddings always match.

In [5]:
# Part 2 loads exactly this path and collection name
store = GameVectorStore(path="chromadb", collection_name="udaplay")
chroma_client = store.client

### Collection

In [6]:
# OpenAI embeddings, reading the key from OPENAI_API_KEY (not CHROMA_OPENAI_API_KEY)
embedding_fn = store.embedding_function
collection = store.collection
print(f"Collection '{collection.name}' ready, embedding function: {type(embedding_fn).__name__}")

Collection 'udaplay' ready, embedding function: OpenAIEmbeddingFunction


### Add documents

Each file in `games/` becomes one document, with the file name (like `001`) as id and the game fields as metadata. The embedded text includes platform, name, year, genre, publisher and description. `load_games()` uses upsert, so re-running this notebook does not create duplicates.

In [7]:
# Make sure you have a directory "project/starter/games"
indexed = store.load_games("games")
print(f"Indexed {indexed} games, documents in collection: {store.count()}")

example = collection.get(ids=["001"], include=["documents", "metadatas"])
print("\nExample document:", example["documents"][0])
print("Example metadata:", example["metadatas"][0])

Indexed 15 games, documents in collection: 15

Example document: [PlayStation 1] Gran Turismo (1997) - Genre: Racing. Publisher: Sony Computer Entertainment. A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.
Example metadata: {'Publisher': 'Sony Computer Entertainment', 'YearOfRelease': 1997, 'Platform': 'PlayStation 1', 'Name': 'Gran Turismo', 'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.', 'Genre': 'Racing'}


### Semantic search

Query the collection with natural-language questions. A lower distance means a closer match.

In [8]:
def print_results(query: str, n_results: int = 3):
    print(f"Query: {query}")
    for r in store.search(query, n_results):
        print(
            f"  [{r['id']}] {r['Name']} | {r['Platform']} | {r['YearOfRelease']} | "
            f"{r['Publisher']} | distance={r['distance']}"
        )
    print()


queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Who published Grand Theft Auto: San Andreas?",
    "Racing games on PlayStation",
    "Was Mortal Kombat X released for PlayStation 5?",  # not in the dataset
]

for q in queries:
    print_results(q)

Query: When was Pokémon Gold and Silver released?


  [006] Pokémon Gold and Silver | Game Boy Color | 1999 | Nintendo | distance=0.1184
  [007] Pokémon Ruby and Sapphire | Game Boy Advance | 2002 | Nintendo | distance=0.1508
  [009] Super Mario 64 | Nintendo 64 | 1996 | Nintendo | distance=0.2295

Query: Which one was the first 3D platformer Mario game?


  [009] Super Mario 64 | Nintendo 64 | 1996 | Nintendo | distance=0.123
  [008] Super Mario World | Super Nintendo Entertainment System (SNES) | 1990 | Nintendo | distance=0.1437
  [010] Super Smash Bros. Melee | GameCube | 2001 | Nintendo | distance=0.208

Query: Who published Grand Theft Auto: San Andreas?


  [002] Grand Theft Auto: San Andreas | PlayStation 2 | 2004 | Rockstar Games | distance=0.1233
  [001] Gran Turismo | PlayStation 1 | 1997 | Sony Computer Entertainment | distance=0.2101
  [003] Gran Turismo 5 | PlayStation 3 | 2010 | Sony Computer Entertainment | distance=0.2133

Query: Racing games on PlayStation


  [003] Gran Turismo 5 | PlayStation 3 | 2010 | Sony Computer Entertainment | distance=0.1169
  [001] Gran Turismo | PlayStation 1 | 1997 | Sony Computer Entertainment | distance=0.1204
  [012] Mario Kart 8 Deluxe | Nintendo Switch | 2017 | Nintendo | distance=0.1792

Query: Was Mortal Kombat X released for PlayStation 5?


  [005] Marvel's Spider-Man 2 | PlayStation 5 | 2023 | Sony Interactive Entertainment | distance=0.2143
  [004] Marvel's Spider-Man | PlayStation 4 | 2018 | Sony Interactive Entertainment | distance=0.228
  [015] Halo Infinite | Xbox Series X|S | 2021 | Xbox Game Studios | distance=0.2284



### Reload check

Part 2 opens the existing database with `create=False`, which fails if the collection is missing. This cell does the same to confirm the data persisted on disk.

In [9]:
reloaded = GameVectorStore(path="chromadb", collection_name="udaplay", create=False)
print(reloaded)
print(reloaded.search("Halo", n_results=1)[0]["Name"])

GameVectorStore(collection='udaplay', documents=15)


Halo Infinite
